In [ ]:
!pip install implicit
!pip install lightfm-next

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 2.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for implicit: filename=implicit-0.7.2-cp312-cp312-linux_x86_64.whl size=938107 sha256=ce72037f78ccf0b2539a60576835c39a756677e7c20671982db813aac38856fc
  Stored in directory: /root/.cache/pip/wheels/b2/00/4f/9ff8af07a0a53ac6007ea5d739da19cfe147a2df542b6899f8
Successfully built implicit
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 32.7 MB/s eta 0:00:00


In [ ]:
import json
import pandas as pd
import random
from tqdm import tqdm
from google.colab import drive
import matplotlib.pyplot as plt
from scipy.sparse import coo_matrix
# from implicit.als import AlternatingLeastSquares
SEED = 22
drive.mount('/content/drive')
import numpy as np
pd.options.display.max_rows = 50
pd.options.display.max_columns = 50

MessageError: Error: credential propagation was unsuccessful

In [ ]:
csv_file_path = '/content/drive/MyDrive/final_yelp_PA_FL.parquet'
df = pd.read_parquet(csv_file_path)
df

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = df.copy()
df["date"] = pd.to_datetime(df["date"])

MIN_INTERACTIONS_CF = 3
K_TEST = 1

# фильтр пользователей
user_cnt = df.groupby("user_id")["business_id"].count()
eligible_users = user_cnt[user_cnt >= MIN_INTERACTIONS_CF].index

df_cf = df[df["user_id"].isin(eligible_users)].copy()
df_cf = df_cf.sort_values(["user_id", "date"])

# LMROO-1
test_idx = df_cf.groupby("user_id").tail(K_TEST).index
test_df  = df_cf.loc[test_idx].copy()
train_df = df_cf.drop(test_idx).copy()

# распределения
cnt_all = df_cf.groupby("user_id")["business_id"].count()
cnt_train = train_df.groupby("user_id")["business_id"].count()

print("df_cf user interactions describe:")
print(cnt_all.describe(percentiles=[.5,.75,.9,.95,.99]))
print("\ntrain_df user interactions describe:")
print(cnt_train.describe(percentiles=[.5,.75,.9,.95,.99]))

# гистограмма (лучше в log-y, т.к. long tail)
plt.figure(figsize=(10,4))
plt.hist(cnt_all.values, bins=50)
plt.title("User interactions count (df_cf, after >=5 filter)")
plt.xlabel("#interactions per user")
plt.ylabel("count of users")
plt.yscale("log")
plt.grid(True)
plt.show()

plt.figure(figsize=(10,4))
plt.hist(cnt_train.values, bins=50)
plt.title("User interactions count (train_df, after LMROO)")
plt.xlabel("#interactions per user in train")
plt.ylabel("count of users")
plt.yscale("log")
plt.grid(True)
plt.show()

# CDF (очень удобно для отчёта)
vals = np.sort(cnt_all.values)
cdf = np.arange(1, len(vals)+1)/len(vals)

plt.figure(figsize=(10,4))
plt.plot(vals, cdf)
plt.title("CDF: user interactions count (df_cf)")
plt.xlabel("#interactions per user")
plt.ylabel("CDF")
plt.grid(True)
plt.show()


In [ ]:
import numpy as np
import pandas as pd

from scipy.sparse import coo_matrix
from implicit.nearest_neighbours import CosineRecommender

np.random.seed(42)

MIN_USER_INTERACTIONS = 3   # фильтр пользователей
MIN_ITEM_INTERACTIONS = 3    # фильтр бизнесов (только по train)
K_TEST = 1                   # LMROO-1

KS = (5, 10, 20, 50)


In [ ]:
df = df.copy()
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["user_id", "date"])

# фильтр пользователей по полному df (это ок, это часть дизайна датасета)
user_cnt = df.groupby("user_id")["business_id"].count()
eligible_users = user_cnt[user_cnt >= MIN_USER_INTERACTIONS].index

df_cf = df[df["user_id"].isin(eligible_users)].copy()
df_cf = df_cf.sort_values(["user_id", "date"])

# LMROO-1: последнее действие каждого пользователя в test
test_idx = df_cf.groupby("user_id").tail(K_TEST).index
test_df  = df_cf.loc[test_idx].copy()
train_df = df_cf.drop(test_idx).copy()

print("Users (after filter):", df_cf["user_id"].nunique())
print("Train interactions:", len(train_df))
print("Test interactions :", len(test_df))
print("Businesses in train (before item-filter):", train_df["business_id"].nunique())
print("Businesses in test  (raw):", test_df["business_id"].nunique())


In [ ]:
# считаем только по train — без утечки
item_cnt = train_df.groupby("business_id")["user_id"].count()
eligible_items = item_cnt[item_cnt >= MIN_ITEM_INTERACTIONS].index

train_df_f = train_df[train_df["business_id"].isin(eligible_items)].copy()
test_df_f  = test_df[test_df["business_id"].isin(eligible_items)].copy()

print("Items before filter:", train_df["business_id"].nunique())
print("Items after  filter:", train_df_f["business_id"].nunique())
print("Train interactions after item-filter:", len(train_df_f))
print("Test interactions  after item-filter:", len(test_df_f))


In [ ]:
# индексация по train_df_f
users = train_df_f["user_id"].unique()
items = train_df_f["business_id"].unique()

user2idx = {u: i for i, u in enumerate(users)}
item2idx = {b: i for i, b in enumerate(items)}

train_df_f["u"] = train_df_f["user_id"].map(user2idx).astype("int32")
train_df_f["i"] = train_df_f["business_id"].map(item2idx).astype("int32")

test_df_f["u"] = test_df_f["user_id"].map(user2idx)
test_df_f["i"] = test_df_f["business_id"].map(item2idx)

# убираем cold-start: юзер/айтем не в train после фильтра
test_df_f = test_df_f.dropna(subset=["u", "i"]).copy()
test_df_f["u"] = test_df_f["u"].astype("int32")
test_df_f["i"] = test_df_f["i"].astype("int32")

n_users = len(users)
n_items = len(items)

R = coo_matrix(
    (np.ones(len(train_df_f), dtype=np.float32),
     (train_df_f["u"].values, train_df_f["i"].values)),
    shape=(n_users, n_items)
).tocsr()

print("R shape:", R.shape, "nnz:", R.nnz)
print("Final test size:", len(test_df_f))


In [ ]:
def recall_at_k(rank, k):
    return 1.0 if rank < k else 0.0

def ndcg_at_k(rank, k):
    if rank >= k:
        return 0.0
    return 1.0 / np.log2(rank + 2)

def map_at_k(rank, k):
    if rank >= k:
        return 0.0
    return 1.0 / (rank + 1)


In [ ]:
# what user has seen in train (чтобы не рекомендовать seen)
user_train_items = train_df_f.groupby("u")["i"].apply(set).to_dict()
all_items = np.arange(n_items, dtype=np.int32)

def evaluate_recommender(test_df_eval, recommend_fn, ks=KS):
    results = {K: {"rec": [], "ndcg": [], "map": []} for K in ks}
    maxK = max(ks)

    short_cnt = 0
    total = 0

    for _, row in test_df_eval.iterrows():
        total += 1
        u = int(row["u"])
        pos = int(row["i"])

        recs = recommend_fn(u, maxK)
        if len(recs) < maxK:
            short_cnt += 1

        hits = np.where(np.array(recs) == pos)[0]
        for K in ks:
            if len(hits) == 0 or hits[0] >= K:
                results[K]["rec"].append(0.0)
                results[K]["ndcg"].append(0.0)
                results[K]["map"].append(0.0)
            else:
                rank = int(hits[0])
                results[K]["rec"].append(1.0)
                results[K]["ndcg"].append(ndcg_at_k(rank, K))
                results[K]["map"].append(map_at_k(rank, K))

    print(f"[debug] short rec lists: {short_cnt}/{total} ({short_cnt/total:.3f})")

    out = {}
    for K in ks:
        out[K] = (float(np.mean(results[K]["rec"])),
                  float(np.mean(results[K]["ndcg"])),
                  float(np.mean(results[K]["map"])))
    return out


def print_metrics(name, out, ks=KS):
    print(f"\n=== {name} ===")
    for K in ks:
        rec, ndcg, mp = out[K]
        print(f"K={K:2d} | Recall@{K}: {rec:.4f} | NDCG@{K}: {ndcg:.4f} | MAP@{K}: {mp:.4f}")


In [ ]:
# популярность по train_df_f: сколько уникальных пользователей взаимодействовали с item
item_pop = train_df_f.groupby("i")["u"].nunique().sort_values(ascending=False)
pop_ranked_items = item_pop.index.values

def recommend_pop(u, K):
    seen = user_train_items.get(u, set())
    recs = []
    for it in pop_ranked_items:
        if it not in seen:
            recs.append(it)
            if len(recs) == K:
                break
    return np.array(recs, dtype=np.int32)

out_pop = evaluate_recommender(test_df_f, recommend_pop, ks=KS)
print_metrics("Popularity (filtered users+items)", out_pop)


In [ ]:
k=10 | precision@k=0.0024 | recall@k=0.0239 | ndcg@k=0.0115
k=20 | precision@k=0.0019 | recall@k=0.0381 | ndcg@k=0.0151
k=50 | precision@k=0.0015 | recall@k=0.0763 | ndcg@k=0.0226



In [ ]:
from implicit.nearest_neighbours import CosineRecommender, tfidf_weight, bm25_weight
import pandas as pd
R = R.tocsr()
RT = R.T.tocsr()
knn = CosineRecommender(K=5)   # можно 100/200/500
knn.fit(RT)                    # implicit ждёт item-user

def recommend_itemknn(u, K):
    rec_items, _ = knn.recommend(
        userid=u,
        user_items=R,
        N=K,
        filter_already_liked_items=True
    )
    return rec_items

out_knn = evaluate_recommender(test_df_f, recommend_itemknn, ks=KS)
print_metrics("ItemKNN (filtered users+items)", out_knn)


In [ ]:
from implicit.nearest_neighbours import CosineRecommender, tfidf_weight, bm25_weight
import pandas as pd

R = R.tocsr()

mats = {
    "raw": R,
    "tfidf": tfidf_weight(R).tocsr(),
    "bm25": bm25_weight(R, K1=1.2, B=0.75).tocsr(),
}

Ks_neighbors = [3, 5, 7, 10, 15]
rows = []

for name, Rw in mats.items():
    RTw = Rw.T.tocsr()
    for K_sim in Ks_neighbors:
        model = CosineRecommender(K=K_sim)
        model.fit(RTw)

        def rec(u, K):
            rec_items, _ = model.recommend(
                userid=u,
                user_items=Rw,
                N=K,
                filter_already_liked_items=True
            )
            return rec_items

        out = evaluate_recommender(test_df_f, rec, ks=(50,))
        rec50, ndcg50, map50 = out[50]
        rows.append((name, K_sim, rec50, ndcg50, map50))

res = pd.DataFrame(rows, columns=["weighting", "K_sim", "Recall@50", "NDCG@50", "MAP@50"])
res.sort_values("Recall@50", ascending=False).head(15)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from implicit.nearest_neighbours import CosineRecommender, tfidf_weight, bm25_weight

# Убедитесь, что R и test_df_f определены
R = R.tocsr()

# Подготовка матриц
mats = {
    "raw": R,
    "tfidf": tfidf_weight(R).tocsr(),
    "bm25": bm25_weight(R, K1=1.2, B=0.75).tocsr(),
}

# Диапазон K_sim
Ks_neighbors = list(range(1, 101, 4))  # от 1 до 100 включительно

# Словари для хранения результатов: {weighting: ([Ks], [Recall], [NDCG], [MAP])}
results = {
    name: {
        'K': [],
        'Recall@50': [],
        'NDCG@50': [],
        'MAP@50': []
    }
    for name in mats.keys()
}

print("Запуск перебора K_sim от 1 до 100...")

for name, Rw in mats.items():
    print(f"\nОбработка веса: {name}")
    RTw = Rw.T.tocsr()

    for K_sim in Ks_neighbors:
        try:
            model = CosineRecommender(K=K_sim)
            model.fit(RTw)

            def rec(u, K):
                rec_items, _ = model.recommend(
                    userid=u,
                    user_items=Rw,
                    N=K,
                    filter_already_liked_items=True
                )
                return rec_items

            # Оценка только для K=50
            out = evaluate_recommender(test_df_f, rec, ks=(50,))
            rec50, ndcg50, map50 = out[50]

            # Сохраняем
            results[name]['K'].append(K_sim)
            results[name]['Recall@50'].append(rec50)
            results[name]['NDCG@50'].append(ndcg50)
            results[name]['MAP@50'].append(map50)

        except Exception as e:
            print(f"  Ошибка при K_sim={K_sim}: {e}")
            # Заполняем NaN, чтобы не нарушать длину
            results[name]['K'].append(K_sim)
            results[name]['Recall@50'].append(np.nan)
            results[name]['NDCG@50'].append(np.nan)
            results[name]['MAP@50'].append(np.nan)

print("Перебор завершён. Строим графики...")

# Настройка стиля
plt.style.use('seaborn-v0_8-whitegrid')
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics = ['Recall@50', 'NDCG@50', 'MAP@50']
titles = ['Recall@50', 'NDCG@50', 'MAP@50']

for ax, metric, title in zip(axes, metrics, titles):
    for name in mats.keys():
        K_vals = results[name]['K']
        y_vals = results[name][metric]
        ax.plot(K_vals, y_vals, label=name, linewidth=2)

    ax.set_xlabel('K_sim (число соседей)')
    ax.set_ylabel(metric)
    ax.set_title(title)
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

# Опционально: сохранить
# plt.savefig("itemknn_k_sim_analysis.png", dpi=200, bbox_inches='tight')

In [ ]:
def recommend_random(u, K):
    seen = user_train_items.get(u, set())
    candidates = np.setdiff1d(all_items, np.fromiter(seen, dtype=np.int32), assume_unique=False)
    if len(candidates) <= K:
        return candidates
    return np.random.choice(candidates, size=K, replace=False)

out_rand = evaluate_recommender(test_df_f, recommend_random, ks=KS)
print_metrics("Random baseline (filtered users+items)", out_rand)


In [ ]:
print("n_users:", n_users, "n_items:", n_items, "train nnz:", R.nnz)
print("test size:", len(test_df_f))


n_users: 34806 n_items: 18913 train nnz: 389271
test size: 31780


In [ ]:
from scipy.sparse import coo_matrix

n_users = train_df_f["u"].nunique()
n_items = train_df_f["i"].nunique()

R_train = coo_matrix(
    (np.ones(len(train_df_f), dtype=np.float32),
     (train_df_f["u"].values, train_df_f["i"].values)),
    shape=(n_users, n_items)
).tocsr()

RT_train = R_train.T.tocsr()

print("R_train:", R_train.shape, "RT_train:", RT_train.shape)
print("max u in train:", train_df_f["u"].max(), "max i in train:", train_df_f["i"].max())
print("max u in test :", test_df_f["u"].max(), "max i in test :", test_df_f["i"].max())


R_train: (34806, 18913) RT_train: (18913, 34806)
max u in train: 34805 max i in train: 18912
max u in test : 34805 max i in test : 18909


## ALS

In [ ]:
import pandas as pd
import numpy as np
from implicit.als import AlternatingLeastSquares

# Убедитесь, что R — CSR
R = R.tocsr()

print("Обучение ALS...")
als_model = AlternatingLeastSquares(
    factors=64,
    regularization=0.005,
    iterations=40,
    alpha=40,
    random_state=SEED,
    dtype=np.float32
)

als_model.fit(R.T)  # item-user matrix

# ПРАВИЛЬНАЯ функция рекомендации
def recommend_als(u, K):
    # user_vec — sparse row vector for user u
    user_vec = R[u]  # shape = (1, n_items)
    rec_items, _ = als_model.recommend(
        userid=0,  # не используется, так как передан user_vec
        user_items=user_vec,
        N=K,
        filter_already_liked_items=True
    )
    return rec_items

# Фильтрация (на всякий случай)
train_user_ids = set(range(R.shape[0]))
test_df_f_filtered = test_df_f[test_df_f['u'].isin(train_user_ids)]

print(f"Тест до фильтрации: {len(test_df_f)}")
print(f"Тест после фильтрации: {len(test_df_f_filtered)}")

if len(test_df_f_filtered) == 0:
    print("❗ Нет общих пользователей!")
else:
    print("Оценка ALS...")
    out_als = evaluate_recommender(test_df_f_filtered, recommend_als, ks=KS)
    print_metrics("ALS", out_als)
    # rec50, ndcg50, map50 = out_als[50]

    # print("\n" + "="*50)
    # print("Результаты ALS:")
    # print("="*50)
    # print(f"Recall@50 : {rec50:.6f}")
    # print(f"NDCG@50   : {ndcg50:.6f}")
    # print(f"MAP@50    : {map50:.6f}")

Обучение ALS...


/usr/local/lib/python3.12/dist-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed csc_matrix instead. Converting to CSR took 0.009258270263671875 seconds
  warnings.warn(


  0%|          | 0/40 [00:00<?, ?it/s]

Тест до фильтрации: 31780
Тест после фильтрации: 31780
Оценка ALS...
[debug] short rec lists: 0/31780 (0.000)

=== ALS ===
K= 5 | Recall@5: 0.0002 | NDCG@5: 0.0001 | MAP@5: 0.0000
K=10 | Recall@10: 0.0002 | NDCG@10: 0.0001 | MAP@10: 0.0001
K=20 | Recall@20: 0.0006 | NDCG@20: 0.0002 | MAP@20: 0.0001
K=50 | Recall@50: 0.0011 | NDCG@50: 0.0003 | MAP@50: 0.0001


In [ ]:
import pandas as pd
import numpy as np
from implicit.als import AlternatingLeastSquares
from itertools import product

# Убедитесь, что определены: R, test_df_f, KS, SEED
R = R.tocsr()
train_user_ids = set(range(R.shape[0]))
test_df_f_filtered = test_df_f[test_df_f['u'].isin(train_user_ids)]

print(f"Тест после фильтрации: {len(test_df_f_filtered)}")

# Сетка гиперпараметров
param_grid = {
    'factors': [32, 64, 128],
    'regularization': [0.001, 0.005, 0.01],
    'alpha': [20, 40, 80]
}

results_als = []

print("Запуск грид-серча для ALS...")
for i, (f, reg, a) in enumerate(product(param_grid['factors'],
                                        param_grid['regularization'],
                                        param_grid['alpha']), 1):
    print(f"\n[{i}/{len(list(product(*param_grid.values())))}] "
          f"factors={f}, reg={reg}, alpha={a}")

    try:
        model = AlternatingLeastSquares(
            factors=f,
            regularization=reg,
            iterations=40,
            alpha=a,
            random_state=SEED,
            dtype=np.float32
        )
        model.fit(R.T)

        def rec_als(u, K):
            user_vec = R[u]
            items, _ = model.recommend(
                userid=0,
                user_items=user_vec,
                N=K,
                filter_already_liked_items=True
            )
            return items

        out = evaluate_recommender(test_df_f_filtered, rec_als, ks=KS)
        rec10, ndcg10, map10 = out[10]

        results_als.append({
            'factors': f,
            'regularization': reg,
            'alpha': a,
            'Recall@10': rec10,
            'NDCG@10': ndcg10,
            'MAP@10': map10
        })

        print(f"  → Recall@10 = {rec10:.6f}")

    except Exception as e:
        print(f"  ❌ Ошибка: {e}")
        results_als.append({
            'factors': f,
            'regularization': reg,
            'alpha': a,
            'Recall@10': np.nan,
            'NDCG@10': np.nan,
            'MAP@10': np.nan
        })

# Преобразуем в DataFrame
df_als = pd.DataFrame(results_als)
df_als.to_csv("als_grid_search.csv", index=False)

# Лучшая модель по Recall@50
best_row = df_als.loc[df_als['Recall@10'].idxmax()]
print("\n" + "="*60)
print("ЛУЧШИЕ ГИПЕРПАРАМЕТРЫ ALS (по Recall@10):")
print("="*60)
print(f"factors        = {best_row['factors']}")
print(f"regularization = {best_row['regularization']}")
print(f"alpha          = {best_row['alpha']}")
print(f"Recall@50      = {best_row['Recall@10']:.6f}")
print(f"NDCG@50        = {best_row['NDCG@10']:.6f}")
print(f"MAP@50         = {best_row['MAP@10']:.6f}")

## LightFM

In [ ]:
import re
import ast
import pandas as pd
import numpy as np

def safe_parse_dict(x):
    if pd.isna(x):
        return {}
    if isinstance(x, dict):
        return x
    if isinstance(x, str):
        s = x.strip()
        if s == "" or s.lower() in {"none", "nan"}:
            return {}
        try:
            d = ast.literal_eval(s)
            return d if isinstance(d, dict) else {}
        except Exception:
            return {}
    return {}

def coerce_bool(v):
    """Пытаемся привести строковые True/False к bool. Иначе вернуть как есть."""
    if isinstance(v, bool):
        return v
    if isinstance(v, (int, np.integer)) and v in (0, 1):
        return bool(v)
    if isinstance(v, str):
        s = v.strip().lower()
        if s in {"true", "t", "yes", "y", "1"}:
            return True
        if s in {"false", "f", "no", "n", "0"}:
            return False
    return v

def try_parse_nested(v):
    if isinstance(v, str):
        s = v.strip()
        if (s.startswith("{") and s.endswith("}")) or (s.startswith("[") and s.endswith("]")):
            try:
                return ast.literal_eval(s)
            except Exception:
                return v
    return v

def flatten_attributes(attr_dict, prefix="attr"):
    feats = []
    for k, v in attr_dict.items():
        if v is None:
            continue

        v = try_parse_nested(v)
        v = coerce_bool(v)

        # nested dict
        if isinstance(v, dict):
            for kk, vv in v.items():
                if vv is None:
                    continue
                vv = try_parse_nested(vv)
                vv = coerce_bool(vv)

                if isinstance(vv, bool):
                    if vv:  # добавляем только True
                        feats.append(f"{prefix}:{k}.{kk}=1")
                else:
                    vv_str = str(vv).strip().replace(" ", "_")
                    if vv_str and vv_str.lower() not in {"none", "nan"}:
                        feats.append(f"{prefix}:{k}.{kk}={vv_str}")
            continue

        # bool
        if isinstance(v, bool):
            if v:  # только True
                feats.append(f"{prefix}:{k}=1")
            continue

        # string/number
        v_str = str(v).strip()
        if v_str == "" or v_str.lower() in {"none", "nan"}:
            continue
        v_str = v_str.replace(" ", "_")
        feats.append(f"{prefix}:{k}={v_str}")

    return feats

def split_categories(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        cats = x
    else:
        cats = [c.strip() for c in str(x).split(",")]
    cats = [c for c in cats if c and c.lower() not in {"none", "nan"}]
    return [f"cat:{c.replace(' ', '_')}" for c in cats]

In [ ]:
import numpy as np
import pandas as pd
import ast
import scipy.sparse as sp

from lightfm import LightFM
from lightfm.data import Dataset
from lightfm.cross_validation import random_train_test_split
from lightfm.evaluation import precision_at_k, recall_at_k

# -----------------------------
# 1) helpers: parsing & featureization
# -----------------------------
df = df.sort_values(["user_id", "business_id", "date"])
df = df.drop_duplicates(["user_id", "business_id"], keep="last")
def safe_parse_dict(x):
    """
    attributes в Yelp часто строка вида "{'WiFi': 'free', 'BikeParking': True, ...}"
    Иногда None/NaN/кривые строки.
    """
    if pd.isna(x):
        return {}
    if isinstance(x, dict):
        return x
    if isinstance(x, str):
        s = x.strip()
        if s == "" or s.lower() == "none":
            return {}
        try:
            d = ast.literal_eval(s)
            return d if isinstance(d, dict) else {}
        except Exception:
            return {}
    return {}

def try_parse_nested(v):
    # если строка выглядит как dict/list - пробуем распарсить
    if isinstance(v, str):
        s = v.strip()
        if (s.startswith("{") and s.endswith("}")) or (s.startswith("[") and s.endswith("]")):
            try:
                return ast.literal_eval(s)
            except Exception:
                return v
    return v

def flatten_attributes(attr_dict, prefix="attr"):
    feats = []
    for k, v in attr_dict.items():
        if v is None:
            continue

        v = try_parse_nested(v)

        # вложенный dict
        if isinstance(v, dict):
            for kk, vv in v.items():
                if vv is None:
                    continue
                vv = try_parse_nested(vv)
                if isinstance(vv, bool):
                    if vv:
                        feats.append(f"{prefix}:{k}.{kk}=1")
                else:
                    vv_str = str(vv).strip().replace(" ", "_")
                    if vv_str and vv_str.lower() != "none":
                        feats.append(f"{prefix}:{k}.{kk}={vv_str}")
            continue

        # bool
        if isinstance(v, bool):
            if v:
                feats.append(f"{prefix}:{k}=1")
            continue

        # string/number
        v_str = str(v).strip()
        if v_str == "" or v_str.lower() == "none":
            continue
        v_str = v_str.replace(" ", "_")
        feats.append(f"{prefix}:{k}={v_str}")

    return feats

def bin_numeric(series, n_bins=10, prefix="num"):
    """
    Бинируем числовой столбец в квантильные бины => категориальные токены.
    Так проще и стабильнее для LightFM.
    """
    s = pd.to_numeric(series, errors="coerce")
    # если мало уникальных значений — qcut может упасть
    try:
        b = pd.qcut(s, q=n_bins, duplicates="drop")
    except Exception:
        b = pd.cut(s, bins=min(n_bins, s.nunique()), duplicates="drop")
    return b.astype(str).fillna("nan").map(lambda z: f"{prefix}:{z.replace(' ', '')}")

def parse_elite(x):
    """
    elite у Yelp бывает строка годов "2019,2020" или None.
    Превращаем в bool: elite_any
    """
    if pd.isna(x):
        return False
    if isinstance(x, (list, tuple, set)):
        return len(x) > 0
    s = str(x).strip()
    if s == "" or s.lower() == "none":
        return False
    # если это "None" или "nan" уже отсеяли; иначе считаем что есть
    return True

# -----------------------------
# 2) main pipeline
# -----------------------------

# !!! df должен быть твоим датафреймом
df = df.copy()

# даты
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["user_id", "business_id", "date"])

# (A) фильтры по >=5 интеракций
min_user_inter = 10
min_item_inter = 10

user_cnt = df["user_id"].value_counts()
item_cnt = df["business_id"].value_counts()

df = df[df["user_id"].isin(user_cnt[user_cnt >= min_user_inter].index)]
df = df[df["business_id"].isin(item_cnt[item_cnt >= min_item_inter].index)]

# (B) next-item split: у каждого юзера последний item -> test, остальное -> train
df = df.sort_values(["user_id", "date"])
df["rank_desc"] = df.groupby("user_id").cumcount(ascending=True)
# индекс последнего
last_idx = df.groupby("user_id")["date"].idxmax()

test_df = df.loc[last_idx, ["user_id", "business_id", "date"]].copy()
train_df = df.drop(index=last_idx).copy()

# (опционально) если у кого-то осталась 0 история в train — выкидываем их из теста
train_users = set(train_df["user_id"].unique())
test_df = test_df[test_df["user_id"].isin(train_users)].copy()

# -----------------------------
# 3) build user/item feature tables (aggregated)
# -----------------------------

# item static features (берём по business_id первое/последнее ненулевое)
item_cols = [
    "business_id", "is_open", "attributes", "categories", "city", "state",
    "latitude", "longitude", "stars_business", "review_count"
]
item_df = (df[item_cols]
           .drop_duplicates("business_id")
           .set_index("business_id")
           .copy())

# user static features (берём по user_id)
user_cols = [
    "user_id", "review_count_user", "yelping_since", "fans",
    "average_stars", "elite", "compliment_count"
]
user_df = (df[user_cols]
           .drop_duplicates("user_id")
           .set_index("user_id")
           .copy())

# yelping_since -> стаж в днях -> бины
user_df["yelping_since"] = pd.to_datetime(user_df["yelping_since"], errors="coerce")
user_df["tenure_days"] = (df["date"].max() - user_df["yelping_since"]).dt.days

# -----------------------------
# 4) Create tokenized feature lists
# -----------------------------

# item features
item_features = {}
# предварительно сделаем бины для числовых item полей
item_df["lat_bin"] = bin_numeric(item_df["latitude"], n_bins=20, prefix="lat")
item_df["lon_bin"] = bin_numeric(item_df["longitude"], n_bins=20, prefix="lon")
item_df["stars_bin"] = bin_numeric(item_df["stars_business"], n_bins=10, prefix="stars_biz")
item_df["rc_bin"] = bin_numeric(item_df["review_count"], n_bins=10, prefix="biz_rc")

for bid, row in item_df.iterrows():
    feats = []

    # is_open
    if "is_open" in row and not pd.isna(row["is_open"]):
        feats.append(f"is_open:{int(row['is_open'])}")

    # city/state (можно оставить только state если боишься переобучения)
    if not pd.isna(row.get("state", np.nan)):
        feats.append(f"state:{str(row['state']).strip()}")
    if not pd.isna(row.get("city", np.nan)):
        feats.append(f"city:{str(row['city']).strip().replace(' ', '_')}")

    # categories
    feats += split_categories(row.get("categories", None))

    # attributes
    ad = safe_parse_dict(row.get("attributes", None))
    feats += flatten_attributes(ad, prefix="attr")

    # numeric bins
    feats.append(row["lat_bin"])
    feats.append(row["lon_bin"])
    feats.append(row["stars_bin"])
    feats.append(row["rc_bin"])

    item_features[bid] = feats

# user features
user_features = {}
user_df["tenure_bin"] = bin_numeric(user_df["tenure_days"], n_bins=10, prefix="tenure")
user_df["u_rc_bin"] = bin_numeric(user_df["review_count_user"], n_bins=10, prefix="user_rc")
user_df["fans_bin"] = bin_numeric(user_df["fans"], n_bins=10, prefix="fans")
user_df["avg_stars_bin"] = bin_numeric(user_df["average_stars"], n_bins=10, prefix="user_avgstars")
user_df["compl_bin"] = bin_numeric(user_df["compliment_count"], n_bins=10, prefix="compl")

for uid, row in user_df.iterrows():
    feats = []
    feats.append(row["tenure_bin"])
    feats.append(row["u_rc_bin"])
    feats.append(row["fans_bin"])
    feats.append(row["avg_stars_bin"])
    feats.append(row["compl_bin"])

    elite_any = parse_elite(row.get("elite", None))
    feats.append(f"elite:{int(elite_any)}")

    user_features[uid] = feats




In [ ]:
len(df)

219885

In [ ]:
def bin_numeric(series, n_bins=10, prefix="num"):
    s = pd.to_numeric(series, errors="coerce")
    if s.notna().sum() == 0:
        return pd.Series([f"{prefix}:nan"] * len(s), index=s.index)

    try:
        b = pd.qcut(s, q=n_bins, duplicates="drop")
    except Exception:
        nb = min(n_bins, int(s.nunique())) if s.nunique() > 1 else 1
        b = pd.cut(s, bins=nb, duplicates="drop")

    return b.astype(str).fillna("nan").map(lambda z: f"{prefix}:{z.replace(' ', '')}")


In [ ]:
for bid, row in item_df.iterrows():
    feats = []

    feats.append(f"is_open:{int(row['is_open'])}" if not pd.isna(row.get("is_open")) else "is_open:unknown")
    if not pd.isna(row.get("state")):
        feats.append(f"state:{str(row['state']).strip()}")
    if not pd.isna(row.get("city")):
        feats.append(f"city:{str(row['city']).strip().replace(' ', '_')}")

    feats += split_categories(row.get("categories"))
    feats += flatten_attributes(safe_parse_dict(row.get("attributes")), prefix="attr")

    feats += [row["lat_bin"], row["lon_bin"], row["stars_bin"], row["rc_bin"]]

    # дедуп + удалим пустые/None
    feats = [f for f in dict.fromkeys(feats) if isinstance(f, str) and f != ""]
    item_features[bid] = feats


In [ ]:
# 1) Нет ли False/True строками
bad_truefalse = [t for feats in item_features.values() for t in feats if "=True" in t or "=False" in t]
print("tokens with =True/=False:", len(bad_truefalse))

# 2) Не пустые
print("empty item feats:", sum(len(v)==0 for v in item_features.values()))
print("empty user feats:", sum(len(v)==0 for v in user_features.values()))

# 3) Словарь не взорвался
unique_item = len(set(t for feats in item_features.values() for t in feats))
unique_user = len(set(t for feats in user_features.values() for t in feats))
print("unique item tokens:", unique_item)
print("unique user tokens:", unique_user)


tokens with =True/=False: 89881
empty item feats: 0
empty user feats: 0
unique item tokens: 1145
unique user tokens: 44


In [ ]:
# 1) размеры
print("n_users in user_features:", len(user_features))
print("n_items in item_features:", len(item_features))

# 2) примеры (посмотри глазами)
some_u = next(iter(user_features))
some_i = next(iter(item_features))
print("sample user:", some_u, "->", user_features[some_u][:20])
print("sample item:", some_i, "->", item_features[some_i][:20])

# 3) нет ли пустых списков
empty_users = [u for u, feats in user_features.items() if (feats is None or len(feats)==0)]
empty_items = [i for i, feats in item_features.items() if (feats is None or len(feats)==0)]
print("empty user feature rows:", len(empty_users))
print("empty item feature rows:", len(empty_items))

# 4) токены должны быть строками и без nan
def check_tokens(d, name, n=5):
    bad = []
    for k, feats in d.items():
        for f in feats:
            if not isinstance(f, str) or f.strip()=="" or "nan" == f.lower():
                bad.append((k, f))
                if len(bad) >= n:
                    break
        if len(bad) >= n:
            break
    print(f"{name}: bad examples:", bad[:n])

check_tokens(user_features, "user_features")
check_tokens(item_features, "item_features")

# 5) доля уникальных токенов (примерно)
all_u_tokens = set(t for feats in user_features.values() for t in feats)
all_i_tokens = set(t for feats in item_features.values() for t in feats)
print("unique user feature tokens:", len(all_u_tokens))
print("unique item feature tokens:", len(all_i_tokens))


n_users in user_features: 10740
n_items in item_features: 8375
sample user: --4AjktZiHowEIBCMd4CZA -> ['tenure:(2012.9,2462.8]', 'user_rc:(56.0,68.0]', 'fans:(-0.001,1.0]', 'user_avgstars:(3.98,4.1]', 'compl:(-0.001,1.0]', 'elite:0']
sample item: wm5mQ4cSpvko9WlCq07RFw -> ['is_open:1', 'state:PA', 'city:Conshohocken', 'cat:Mexican', 'cat:Restaurants', "attr:Alcohol=u'none'", 'attr:Ambience.classy=1', 'attr:Ambience.casual=1', 'attr:BYOB=True', 'attr:BikeParking=True', 'attr:BusinessAcceptsCreditCards=True', 'attr:BusinessParking.garage=1', 'attr:BusinessParking.street=1', 'attr:BusinessParking.lot=1', 'attr:Caters=True', 'attr:Corkage=False', 'attr:DogsAllowed=False', 'attr:GoodForKids=True', 'attr:GoodForMeal.lunch=1', 'attr:GoodForMeal.dinner=1']
empty user feature rows: 0
empty item feature rows: 0
user_features: bad examples: []
item_features: bad examples: []
unique user feature tokens: 44
unique item feature tokens: 1145


In [ ]:
n_intersections = test_interactions.multiply(train_interactions).nnz
print("intersections:", n_intersections)  # должно быть 0


intersections: 410


In [ ]:
# -----------------------------
# 5) LightFM Dataset: fit + build matrices
# -----------------------------

all_users = train_df["user_id"].unique().tolist()
all_items = df["business_id"].unique().tolist()

# соберём весь словарь фич (иначе Dataset.fit можно и без них, но лучше явно)
all_user_feature_tokens = sorted({f for feats in user_features.values() for f in feats})
all_item_feature_tokens = sorted({f for feats in item_features.values() for f in feats})

dataset = Dataset()
dataset.fit(
    users=all_users,
    items=all_items,
    user_features=all_user_feature_tokens,
    item_features=all_item_feature_tokens
)

# interactions (implicit: 1)
train_interactions, train_weights = dataset.build_interactions(
    (row.user_id, row.business_id, 1.0) for row in train_df.itertuples(index=False)
)

# test interactions: по одному item на user
test_interactions, _ = dataset.build_interactions(
    (row.user_id, row.business_id, 1.0) for row in test_df.itertuples(index=False)
)

# build feature matrices
user_features_mat = dataset.build_user_features(
    ((uid, feats) for uid, feats in user_features.items()),
    normalize=True
)
item_features_mat = dataset.build_item_features(
    ((iid, feats) for iid, feats in item_features.items()),
    normalize=True
)

In [ ]:
model = LightFM(
    no_components=64,
    loss="warp",          # часто топ для top-k имплицитной рекомендации
    learning_rate=0.05,
    item_alpha=1e-6,
    user_alpha=1e-6,
    random_state=42
)

model.fit(
    train_interactions,
    user_features=user_features_mat,
    item_features=item_features_mat,
    epochs=30,
    num_threads=8,
    verbose=True
)


Epoch: 100%|██████████| 30/30 [03:37<00:00,  7.26s/it]


In [ ]:
Ks = [10, 20, 50]

prec = {}
rec = {}

for k in Ks:
    prec[k] = float(precision_at_k(
        model,
        test_interactions,
        train_interactions=train_interactions,
        k=k,
        user_features=user_features_mat,
        item_features=item_features_mat,
        num_threads=8
    ).mean())

    rec[k] = float(recall_at_k(
        model,
        test_interactions,
        train_interactions=train_interactions,
        k=k,
        user_features=user_features_mat,
        item_features=item_features_mat,
        num_threads=8
    ).mean())

# ---- ndcg@k for "one held-out item per user"
# ndcg@k = 1/log2(rank+1) if test item in top-k else 0, since ideal DCG = 1
user_id_map, user_feature_map, item_id_map, item_feature_map = dataset.mapping()
inv_item_id_map = {v: k for k, v in item_id_map.items()}

# precompute seen items in train per user (internal ids)
train_csr = train_interactions.tocsr()
test_csr = test_interactions.tocsr()

all_item_internal = np.arange(train_interactions.shape[1], dtype=np.int32)

def ndcg_at_k_one_item(model, k):
    ndcgs = []
    for u_internal in range(train_interactions.shape[0]):
        # если в тесте нет таргета (на всякий)
        test_items = test_csr[u_internal].indices
        if len(test_items) == 0:
            continue
        target = test_items[0]

        # кандидаты: все items, но исключаем train-seen
        seen = set(train_csr[u_internal].indices.tolist())
        # можно не строить гигантский список через setdiff1d каждый раз,
        # но для понятности оставим так:
        candidates = np.setdiff1d(all_item_internal, np.fromiter(seen, dtype=np.int32), assume_unique=False)

        scores = model.predict(
            u_internal,
            candidates,
            user_features=user_features_mat,
            item_features=item_features_mat
        )

        # top-k по score
        topk_idx = np.argpartition(-scores, kth=min(k, len(scores)-1))[:k]
        topk_items = candidates[topk_idx]
        # сортируем top-k по убыванию score (для ранга)
        topk_scores = scores[topk_idx]
        order = np.argsort(-topk_scores)
        ranked_items = topk_items[order]

        # rank (1-based)
        hits = np.where(ranked_items == target)[0]
        if len(hits) == 0:
            ndcgs.append(0.0)
        else:
            rank = int(hits[0]) + 1
            ndcgs.append(1.0 / np.log2(rank + 1))
    return float(np.mean(ndcgs)) if ndcgs else 0.0

ndcg = {k: ndcg_at_k_one_item(model, k) for k in Ks}

print("=== Metrics (Next-item) ===")
for k in Ks:
    print(f"k={k:2d} | precision@k={prec[k]:.4f} | recall@k={rec[k]:.4f} | ndcg@k={ndcg[k]:.4f}")


ValueError: Test interactions matrix and train interactions matrix share 410 interactions. This will cause incorrect evaluation, check your data split.

In [ ]:
import numpy as np
import pandas as pd
import ast
import scipy.sparse as sp

from lightfm import LightFM
from lightfm.data import Dataset
from lightfm.evaluation import precision_at_k, recall_at_k


# -----------------------------
# 1) helpers
# -----------------------------
def safe_parse_dict(x):
    if pd.isna(x):
        return {}
    if isinstance(x, dict):
        return x
    if isinstance(x, str):
        s = x.strip()
        if s == "" or s.lower() in {"none", "nan"}:
            return {}
        try:
            d = ast.literal_eval(s)
            return d if isinstance(d, dict) else {}
        except Exception:
            return {}
    return {}

def normalize_scalar(v):
    """приводим 'True'/'False'/'None' к bool/None, остальное оставляем"""
    if isinstance(v, str):
        s = v.strip()
        if s.lower() in {"none", "nan", ""}:
            return None
        if s.lower() == "true":
            return True
        if s.lower() == "false":
            return False
    return v

def try_parse_nested(v):
    v = normalize_scalar(v)
    if isinstance(v, str):
        s = v.strip()
        if (s.startswith("{") and s.endswith("}")) or (s.startswith("[") and s.endswith("]")):
            try:
                vv = ast.literal_eval(s)
                return vv
            except Exception:
                return v
    return v

def flatten_attributes(attr_dict, prefix="attr"):
    feats = []
    for k, v in attr_dict.items():
        if v is None:
            continue
        v = try_parse_nested(v)
        if v is None:
            continue

        # nested dict
        if isinstance(v, dict):
            for kk, vv in v.items():
                vv = try_parse_nested(vv)
                if vv is None:
                    continue
                if isinstance(vv, bool):
                    if vv:
                        feats.append(f"{prefix}:{k}.{kk}=1")
                else:
                    vv_str = str(vv).strip().replace(" ", "_")
                    if vv_str and vv_str.lower() not in {"none", "nan"}:
                        feats.append(f"{prefix}:{k}.{kk}={vv_str}")
            continue

        # bool
        if isinstance(v, bool):
            if v:
                feats.append(f"{prefix}:{k}=1")
            continue

        # scalar -> key=value
        v_str = str(v).strip().replace(" ", "_")
        if v_str and v_str.lower() not in {"none", "nan"}:
            feats.append(f"{prefix}:{k}={v_str}")
    return feats

def split_categories(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        cats = x
    else:
        cats = [c.strip() for c in str(x).split(",")]
    cats = [c for c in cats if c and c.lower() != "none"]
    return [f"cat:{c.replace(' ', '_')}" for c in cats]

def bin_numeric(series, n_bins=10, prefix="num"):
    s = pd.to_numeric(series, errors="coerce")
    try:
        b = pd.qcut(s, q=n_bins, duplicates="drop")
    except Exception:
        nb = min(n_bins, max(1, s.nunique()))
        b = pd.cut(s, bins=nb, duplicates="drop")
    return b.astype(str).fillna("nan").map(lambda z: f"{prefix}:{z.replace(' ', '')}")

def parse_elite_any(x):
    if pd.isna(x):
        return False
    if isinstance(x, (list, tuple, set)):
        return len(x) > 0
    s = str(x).strip()
    return not (s == "" or s.lower() in {"none", "nan"})


# -----------------------------
# 2) PREP: collapse to unique (user,business) with last date
# -----------------------------
df = df.copy()
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["user_id", "business_id", "date"])

# фильтры по >=N интеракций
min_user_inter = 10
min_item_inter = 10

user_cnt = df["user_id"].value_counts()
item_cnt = df["business_id"].value_counts()
df = df[df["user_id"].isin(user_cnt[user_cnt >= min_user_inter].index)]
df = df[df["business_id"].isin(item_cnt[item_cnt >= min_item_inter].index)]

# !!! КЛЮЧЕВО: убираем дубли (user,business), оставляем последний по времени
df = (df.sort_values(["user_id", "business_id", "date"])
        .drop_duplicates(["user_id", "business_id"], keep="last"))

# next-item split: последний item на юзера -> test
df = df.sort_values(["user_id", "date"])
last_idx = df.groupby("user_id")["date"].idxmax()

test_df  = df.loc[last_idx, ["user_id", "business_id"]].copy()
train_df = df.drop(index=last_idx)[["user_id", "business_id"]].copy()

# выкинуть юзеров, у которых train пустой (на всякий)
train_users = set(train_df["user_id"].unique())
test_df = test_df[test_df["user_id"].isin(train_users)].copy()


# -----------------------------
# 3) Build static user/item tables + tokens
# -----------------------------
item_cols = ["business_id","is_open","attributes","categories","city","state",
             "latitude","longitude","stars_business","review_count"]
item_df = (df[item_cols].drop_duplicates("business_id").set_index("business_id"))

user_cols = ["user_id","review_count_user","yelping_since","fans",
             "average_stars","elite","compliment_count"]
user_df = (df[user_cols].drop_duplicates("user_id").set_index("user_id"))

# yelping_since -> tenure_days bins
user_df["yelping_since"] = pd.to_datetime(user_df["yelping_since"], errors="coerce")
user_df["tenure_days"] = (df["date"].max() - user_df["yelping_since"]).dt.days

# bin numerics
item_df["lat_bin"]   = bin_numeric(item_df["latitude"],       n_bins=20, prefix="lat")
item_df["lon_bin"]   = bin_numeric(item_df["longitude"],      n_bins=20, prefix="lon")
item_df["stars_bin"] = bin_numeric(item_df["stars_business"], n_bins=10, prefix="stars_biz")
item_df["rc_bin"]    = bin_numeric(item_df["review_count"],   n_bins=10, prefix="biz_rc")

user_df["tenure_bin"]    = bin_numeric(user_df["tenure_days"],        n_bins=10, prefix="tenure")
user_df["u_rc_bin"]      = bin_numeric(user_df["review_count_user"],  n_bins=10, prefix="user_rc")
user_df["fans_bin"]      = bin_numeric(user_df["fans"],              n_bins=10, prefix="fans")
user_df["avg_stars_bin"] = bin_numeric(user_df["average_stars"],     n_bins=10, prefix="user_avgstars")
user_df["compl_bin"]     = bin_numeric(user_df["compliment_count"],  n_bins=10, prefix="compl")

# tokens dicts
item_features = {}
for bid, row in item_df.iterrows():
    feats = []
    feats.append(f"is_open:{int(row['is_open'])}" if not pd.isna(row["is_open"]) else "is_open:0")
    if not pd.isna(row.get("state", np.nan)): feats.append(f"state:{str(row['state']).strip()}")
    if not pd.isna(row.get("city",  np.nan)): feats.append(f"city:{str(row['city']).strip().replace(' ','_')}")
    feats += split_categories(row.get("categories", None))
    feats += flatten_attributes(safe_parse_dict(row.get("attributes", None)), prefix="attr")
    feats += [row["lat_bin"], row["lon_bin"], row["stars_bin"], row["rc_bin"]]
    item_features[bid] = feats

user_features = {}
for uid, row in user_df.iterrows():
    feats = [row["tenure_bin"], row["u_rc_bin"], row["fans_bin"], row["avg_stars_bin"], row["compl_bin"]]
    feats.append(f"elite:{int(parse_elite_any(row.get('elite', None)))}")
    user_features[uid] = feats


# -----------------------------
# 4) LightFM Dataset -> matrices
# -----------------------------
dataset = Dataset()
dataset.fit(
    users=train_df["user_id"].unique(),
    items=item_df.index.values,
    user_features=set(t for fs in user_features.values() for t in fs),
    item_features=set(t for fs in item_features.values() for t in fs),
)

train_interactions, _ = dataset.build_interactions(
    (row.user_id, row.business_id) for row in train_df.itertuples(index=False)
)
test_interactions, _ = dataset.build_interactions(
    (row.user_id, row.business_id) for row in test_df.itertuples(index=False)
)

user_features_mat = dataset.build_user_features(
    ((u, feats) for u, feats in user_features.items()),
    normalize=True
)
item_features_mat = dataset.build_item_features(
    ((i, feats) for i, feats in item_features.items()),
    normalize=True
)

# ---- check: no intersections
n_intersections = test_interactions.multiply(train_interactions).nnz
print("train/test intersections:", n_intersections)  # должно быть 0


# -----------------------------
# 5) Train
# -----------------------------
model = LightFM(
    no_components=64,
    loss="warp",
    learning_rate=0.05,
    item_alpha=1e-6,
    user_alpha=1e-6,
    random_state=42
)
model.fit(
    train_interactions,
    user_features=user_features_mat,
    item_features=item_features_mat,
    epochs=30,
    num_threads=8,
    verbose=True
)


# -----------------------------
# 6) Metrics: precision/recall + ndcg@k (one held-out item)
# -----------------------------
Ks = [10, 20, 50]

prec, rec = {}, {}
for k in Ks:
    prec[k] = float(precision_at_k(
        model,
        test_interactions,
        train_interactions=train_interactions,
        k=k,
        user_features=user_features_mat,
        item_features=item_features_mat,
        num_threads=8
    ).mean())
    rec[k] = float(recall_at_k(
        model,
        test_interactions,
        train_interactions=train_interactions,
        k=k,
        user_features=user_features_mat,
        item_features=item_features_mat,
        num_threads=8
    ).mean())

train_csr = train_interactions.tocsr()
test_csr  = test_interactions.tocsr()
all_items = np.arange(train_interactions.shape[1], dtype=np.int32)

def ndcg_at_k_one_item(k):
    ndcgs = []
    for u in range(train_interactions.shape[0]):
        tgt = test_csr[u].indices
        if len(tgt) == 0:
            continue
        target = tgt[0]

        seen = train_csr[u].indices
        # кандидаты = все - seen
        mask = np.ones(train_interactions.shape[1], dtype=bool)
        mask[seen] = False
        candidates = all_items[mask]

        scores = model.predict(
            u, candidates,
            user_features=user_features_mat,
            item_features=item_features_mat
        )
        if len(scores) == 0:
            ndcgs.append(0.0)
            continue

        kk = min(k, len(scores))
        topk_idx = np.argpartition(-scores, kk-1)[:kk]
        ranked = candidates[topk_idx][np.argsort(-scores[topk_idx])]

        hit_pos = np.where(ranked == target)[0]
        if len(hit_pos) == 0:
            ndcgs.append(0.0)
        else:
            rank = int(hit_pos[0]) + 1
            ndcgs.append(1.0 / np.log2(rank + 1))
    return float(np.mean(ndcgs)) if ndcgs else 0.0

ndcg = {k: ndcg_at_k_one_item(k) for k in Ks}

print("=== Metrics (Next-item) ===")
for k in Ks:
    print(f"k={k:2d} | precision@k={prec[k]:.4f} | recall@k={rec[k]:.4f} | ndcg@k={ndcg[k]:.4f}")


train/test intersections: 0


Epoch: 100%|██████████| 30/30 [03:29<00:00,  6.97s/it]


=== Metrics (Next-item) ===
k=10 | precision@k=0.0024 | recall@k=0.0239 | ndcg@k=0.0115
k=20 | precision@k=0.0019 | recall@k=0.0381 | ndcg@k=0.0151
k=50 | precision@k=0.0015 | recall@k=0.0763 | ndcg@k=0.0226
